In [1]:
import plotly.express as px
import json
from nnsight import LanguageModel
import json
import numpy as np
import configparser

/venv/geometry-of-truth/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hyperparameters
model = 'llama-3.2-3B-Instruct'
split = 0.8

config = configparser.ConfigParser()
config.read('config.ini')
layer = eval(config[model]['probe_layer'])
noperiod = eval(config[model]['noperiod'])

In [3]:
model = LanguageModel(config[model]['weights_directory'])

with open('experimental_outputs/patching_results.json', 'r') as f:
    out = json.load(f)[-1]
false_prompt = out['false_prompt']
logit_diffs = out['logit_diffs']
n_toks = len(logit_diffs)
model_name = out['model']

# transpose logit_diffs
logit_diffs = [[logit_diffs[i][j] for i in range(0, len(logit_diffs))[::-1]] for j in range(len(logit_diffs[0]))]
probs = [[1 / (1 + np.exp(-logit)) for logit in layer] for layer in logit_diffs]

token_ids = model.tokenizer(false_prompt)['input_ids']
tokens = [
    model.tokenizer.decode([token_id]) + f" ({idx})" for idx, token_id in enumerate(token_ids)
]
tokens = tokens[-n_toks:]

fig = px.imshow(
    logit_diffs,
    x=tokens,
    labels=dict(x="Token", y="Layer"),
    color_continuous_scale='blues',
)
fig.show()

In [ ]:
import json
from collections import defaultdict

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# CONFIG
# -----------------------------
model = LanguageModel(config[model]['weights_directory'])
RESULTS_PATH = "experimental_outputs/patching_results.json"

MODEL_ORDER = [
    "llama-3.2-3B",
    "llama-3.2-3B-Instruct",
]

# -----------------------------
# LOAD + GROUP
# -----------------------------
with open(RESULTS_PATH, "r") as f:
    results = json.load(f)

# prompt -> model -> result
grouped = defaultdict(dict)
for r in results:
    grouped[r["false_prompt"]][r["model"]] = r

prompts = list(grouped.keys())
n_rows = len(prompts)
n_cols = len(MODEL_ORDER)

# -----------------------------
# SUBPLOTS
# -----------------------------
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    column_titles=MODEL_ORDER,
    horizontal_spacing=0.10,
    vertical_spacing=0.06,
)

# -----------------------------
# ADD HEATMAPS
# -----------------------------
for i, prompt in enumerate(prompts):
    for j, model_name in enumerate(MODEL_ORDER):
        if model_name not in grouped[prompt]:
            continue

        out = grouped[prompt][model_name]
        logit_diffs = out["logit_diffs"]

        # transpose to (layers × tokens), reverse layer order
        z = [
            [logit_diffs[ii][jj] for ii in range(len(logit_diffs))[::-1]]
            for jj in range(len(logit_diffs[0]))
        ]

        # token labels
        token_ids = model.tokenizer(prompt)["input_ids"]
        n_toks = len(z[0])
        tokens = [
            model.tokenizer.decode([tid]) + f" ({idx})"
            for idx, tid in enumerate(token_ids)
        ]
        tokens = tokens[-n_toks:]

        # subplot axis index (1-based, row-major)
        ax_idx = i * n_cols + j + 1
        xax = "xaxis" if ax_idx == 1 else f"xaxis{ax_idx}"
        yax = "yaxis" if ax_idx == 1 else f"yaxis{ax_idx}"

        # axis domains (in figure fraction coordinates)
        xdom = fig.layout[xax].domain
        ydom = fig.layout[yax].domain

        # put a small colorbar next to THIS subplot (not all stacked on the right)
        cb_x = xdom[1] + 0.01
        cb_y = (ydom[0] + ydom[1]) / 2
        cb_len = (ydom[1] - ydom[0]) * 0.85

        fig.add_trace(
            go.Heatmap(
                z=z,
                x=tokens,
                colorscale="Blues",
                showscale=True,
                # IMPORTANT: no zmin/zmax -> each subplot autoscale independently
                colorbar=dict(
                    thickness=10,
                    len=cb_len,
                    x=cb_x,
                    y=cb_y,
                ),
            ),
            row=i + 1,
            col=j + 1,
        )

# -----------------------------
# LAYOUT
# -----------------------------
fig.update_layout(
    height=300 * n_rows,
    width=520 * n_cols,
    title="Activation patching heatmaps (rows = prompts, cols = models)",
    font=dict(size=11),
)

fig.show()